In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [10]:
def _get_project_home() -> Path:
    """
    Determine project root directory. Works whether script is run from
    trace-analysis/ or project root.
    """
    # First, check if current working directory is the project root
    cwd = Path.cwd()
    if (cwd / "traces").exists():
        return cwd
    
    # Otherwise, derive from script location
    # Script is at trace-analysis/analyze.py, so project root is parent
    script_dir = Path.cwd()  # In notebook, use cwd
    project_root = script_dir.parent if script_dir.name == "trace-analysis" else script_dir
    
    # Verify traces/ exists
    if (project_root / "traces").exists():
        return project_root
    
    # Fallback: return parent anyway (will fail later with clear error)
    return project_root

PROJECT_HOME = _get_project_home()

def get_csv_path(dataset_number: int) -> Path:
    return (
        PROJECT_HOME
        / "traces"
        / "alibaba"
        / "cluster-trace-microservices-v2022"
        / "data"
        / "CallGraph"
        / f"CallGraph_{dataset_number}.csv"
    )

def _read_one(path: str | Path, **read_csv_kwargs) -> pd.DataFrame:
    return pd.read_csv(path, **read_csv_kwargs)

def _parent_rpc_id(rpc_id: str | float | int | None) -> str | None:
    """
    Return the parent RPC id for a dotted rpc_id string.
    Example: '0.1.2' -> '0.1'. Roots (no dot) return None.
    """
    if rpc_id is None or (isinstance(rpc_id, float) and np.isnan(rpc_id)):
        return None
    rpc_str = str(rpc_id).strip()
    if not rpc_str or "." not in rpc_str:
        return None
    return rpc_str.rsplit(".", 1)[0]

def sample_traces(df: pd.DataFrame, fraction: float, trace_col: str = "traceid", random_state: int | None = None) -> pd.DataFrame:
    """
    Sample a fraction of traces from the dataframe.
    
    For rows belonging to the same trace, either keep them all or drop them all
    (maintains trace integrity).
    
    Args:
        df: Input dataframe with trace data
        fraction: Fraction of traces to sample (0.0 to 1.0)
        trace_col: Column name containing trace IDs (default: "traceid")
        random_state: Random seed for reproducibility (default: None)
    
    Returns:
        DataFrame containing all rows for the sampled traces
    """
    if fraction <= 0.0 or fraction > 1.0:
        raise ValueError(f"fraction must be in (0.0, 1.0], got {fraction}")
    
    if trace_col not in df.columns:
        raise ValueError(f"Column '{trace_col}' not found in dataframe")
    
    # Get unique trace IDs
    unique_traces = df[trace_col].dropna().unique()
    
    if len(unique_traces) == 0:
        print("[WARN] No valid trace IDs found")
        return pd.DataFrame()
    
    # Sample trace IDs
    n_samples = max(1, int(len(unique_traces) * fraction))
    sampled_trace_ids = pd.Series(unique_traces).sample(
        n=n_samples, 
        random_state=random_state
    ).values
    
    # Filter dataframe to keep all rows for sampled traces
    sampled_df = df[df[trace_col].isin(sampled_trace_ids)].copy()
    
    print(f"Sampled {len(sampled_trace_ids):,} traces ({fraction*100:.1f}%) from {len(unique_traces):,} unique traces")
    print(f"Result: {len(sampled_df):,} rows from {len(df):,} original rows")
    
    return sampled_df

def print_sibling_window_overlap_stats(df: pd.DataFrame) -> None:
    """
    Determine whether sibling RPCs (children of the same parent within a trace)
    execute sequentially by checking if their [start, end) windows overlap.
    """
    required_cols = {"traceid", "rpc_id", "timestamp", "rt"}
    missing = required_cols - set(df.columns)
    if missing:
        print(f"[WARN] Cannot compute sibling window stats, missing columns: {sorted(missing)}")
        return

    base = (
        df.loc[:, ["traceid", "rpc_id", "timestamp", "rt"]]
        .dropna(subset=["traceid", "rpc_id", "timestamp", "rt"])
        .copy()
    )
    if base.empty:
        print("[WARN] No rows with complete (traceid, rpc_id, timestamp, rt) data for sibling analysis.")
        return

    base["rpc_id"] = base["rpc_id"].astype(str).str.strip()
    base = base[base["rpc_id"] != ""]
    if base.empty:
        print("[WARN] No valid rpc_id values remain after cleaning for sibling analysis.")
        return

    base["timestamp"] = base["timestamp"].astype(float)
    base["rt"] = base["rt"].astype(float)
    base["end_timestamp"] = base["timestamp"] + base["rt"]
    base["parent_rpc_id"] = base["rpc_id"].map(_parent_rpc_id)

    siblings = base[base["parent_rpc_id"].notna()]
    if siblings.empty:
        print("[INFO] No sibling relationships detected (no parent rpc ids).")
        return

    grouped = siblings.groupby(["traceid", "parent_rpc_id"])

    total_groups = 0
    non_overlap_groups = 0
    overlap_groups = 0
    total_children = 0
    non_overlap_children = 0
    overlap_children = 0
    tol = 1e-9

    for (_, _), group in grouped:
        if len(group) < 2:
            continue
        total_groups += 1
        total_children += len(group)

        ordered = group.sort_values("timestamp")
        prev_end = None
        sequential = True
        for _, row in ordered.iterrows():
            start = float(row["timestamp"])
            end = float(row["end_timestamp"])
            if prev_end is None:
                prev_end = end
                continue
            if start < prev_end - tol:
                sequential = False
                break
            prev_end = max(prev_end, end)

        if sequential:
            non_overlap_groups += 1
            non_overlap_children += len(group)
        else:
            overlap_groups += 1
            overlap_children += len(group)

    if total_groups == 0:
        print("[INFO] No parent nodes have two or more children to compare.")
        return

    pct_groups_seq = (non_overlap_groups / total_groups) * 100.0
    pct_groups_overlap = 100.0 - pct_groups_seq
    pct_children_seq = (non_overlap_children / total_children) * 100.0 if total_children else 0.0
    pct_children_overlap = 100.0 - pct_children_seq

    print("Sibling window overlap:")
    print(f"  Parent nodes with >=2 children: {total_groups:,}")
    print(f"  Sequential (non-overlapping) parents: {non_overlap_groups:,} ({pct_groups_seq:.2f}%)")
    print(f"  Overlapping parents: {overlap_groups:,} ({pct_groups_overlap:.2f}%)")
    print(f"  Child calls under analyzed parents: {total_children:,}")
    print(f"  Children in sequential groups: {non_overlap_children:,} ({pct_children_seq:.2f}%)")
    print(f"  Children in overlapping groups: {overlap_children:,} ({pct_children_overlap:.2f}%)")

In [3]:
# Read one CSV file (dataset 0)
csv_path = get_csv_path(0)
print(f"Reading CSV file: {csv_path}")

df_orig = _read_one(csv_path, on_bad_lines="skip")
print(f"Loaded {len(df_orig)} rows")



Reading CSV file: /mnt/nvme1/proj/masa/traces/alibaba/cluster-trace-microservices-v2022/data/CallGraph/CallGraph_0.csv
Loaded 13331267 rows


In [12]:
# Optional: Sample a fraction of traces to speed up analysis
# Set sample_fraction to 1.0 to use all traces, or a smaller value (e.g., 0.1 for 10%)
sample_fraction = 0.1
if sample_fraction < 1.0:
    df = sample_traces(df_orig, fraction=sample_fraction, random_state=42)
    print()

df.head()

Sampled 214,120 traces (10.0%) from 2,141,207 unique traces
Result: 1,342,531 rows from 13,331,267 original rows



,timestamp,traceid,service,rpc_id,rpctype,um,uminstanceid,interface,dm,dminstanceid,rt
18,168305,T_22121575692,S_85905920,0.1,rpc,MS_23205,MS_23205_POD_1206,1oNt-EK5Lm,MS_24094,MS_24094_POD_4972,3.0
42,56979,T_23776503347,S_73126470,0.1.2.5.33,mc,MS_7226,MS_7226_POD_136,rwvDNgNNj-,MS_14304,MS_14304_POD_139,1.0
45,57271,T_23776503347,S_73126470,0.1.2.9.5.6.1.3370129173,mq,MS_41914,MS_41914_POD_158,b4vXc2doXX,MS_54016,MS_54016_POD_26,1.0
48,56928,T_23776503347,S_73126470,0.1.2.4,mc,MS_39993,MS_39993_POD_83,hHGwtRtkGz,MS_37691,MS_37691_POD_248,1.0
58,41628,T_9925423159,S_18211983,0.1.1.1,mc,MS_29076,MS_29076_POD_110,OctX6xtC2C,MS_39556,MS_39556_POD_148,0.0


In [15]:
# Filter data and run parallel RPC detection
# First, filter out unknowns if needed
df_filtered = df[
    (df["um"].isin(["UNKNOWN", "UNAVAILABLE"]) == False) &
    (df["dm"].isin(["UNKNOWN", "UNAVAILABLE"]) == False)
].copy()

# Select only RPC rows (exclude UNKNOWN and mq types)
rpc_df = df_filtered[~df_filtered["rpctype"].isin(["UNKNOWN", "mq"])].copy()

print(f"Filtered to {len(rpc_df)} RPC rows from {len(df)} total rows")
print("\n" + "="*60)


Filtered to 1046840 RPC rows from 1342531 total rows



In [16]:
# Analyze sibling window overlap statistics
print_sibling_window_overlap_stats(rpc_df)

Sibling window overlap:
  Parent nodes with >=2 children: 158,665
  Sequential (non-overlapping) parents: 75,793 (47.77%)
  Overlapping parents: 82,872 (52.23%)
  Child calls under analyzed parents: 841,916
  Children in sequential groups: 219,480 (26.07%)
  Children in overlapping groups: 622,436 (73.93%)
